# E2 — D-Fire: dados e baseline YOLOv7
Selecione **Runtime > Change runtime type > GPU**. O notebook tenta obter o [espelho Kaggle indicado pelo fornecedor](https://github.com/gaia-solutions-on-demand/DFireDataset) quando não encontra dados preparados no Drive. Confira a origem e a divisão antes do treino. Confirme visualmente a ordem dos IDs das classes. O código executado está no repositório.


In [ ]:
import os, pathlib, subprocess, torch, zipfile
assert torch.cuda.is_available(), 'Ative GPU no ambiente de execução do Colab'
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'  # compatibilidade PyTorch 2.6+ com checkpoint oficial confiável
print(torch.__version__, torch.cuda.get_device_name(0))


## Caminhos
Clone o projeto após enviar esta versão ao GitHub. Para testar alterações ainda locais, envie `smoak-detector-ia.zip` pelo painel Arquivos do Colab; o notebook o extrai em `/content`. Se o repositório for privado, autentique o clone pelo método autorizado pela sua conta; não coloque token no notebook. O ambiente `/content` é temporário; copie os artefatos para seu armazenamento após a execução.


In [ ]:
PROJECT_URL = 'https://github.com/topicos-ia-upe-2026-2/g03-smoak-detector-ai.git'
YOLO_COMMIT = 'a207844b1ce82d204ab36d87d496728d3d2348e7'
DATA_ROOT = pathlib.Path('/content/dfire')
PROJECT = pathlib.Path('/content/smoak-detector-ia')
YOLO = pathlib.Path('/content/yolov7')
LOCAL_ZIP = pathlib.Path('/content/smoak-detector-ia.zip')
if LOCAL_ZIP.exists() and not PROJECT.exists():
    with zipfile.ZipFile(LOCAL_ZIP) as archive: archive.extractall(PROJECT)
if not PROJECT.exists(): subprocess.run(['git', 'clone', PROJECT_URL, str(PROJECT)], check=True)
if not YOLO.exists(): subprocess.run(['git', 'clone', 'https://github.com/WongKinYiu/yolov7.git', str(YOLO)], check=True)
subprocess.run(['git', '-C', str(YOLO), 'checkout', '--detach', YOLO_COMMIT], check=True)
print('Projeto local:', PROJECT, (PROJECT/'src/prepare_dfire.py').exists())
print('YOLOv7:', subprocess.check_output(['git', '-C', str(YOLO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
%pip install -r /content/smoak-detector-ia/requirements-colab.txt
# Torch/torchvision, NumPy e Protobuf são fornecidos pelo runtime do Colab.
# Isso preserva a combinação PyTorch/CUDA e evita pins legados incompatíveis com Python 3.13.
import sys, numpy, google.protobuf
print('Dependências instaladas:', {'python': sys.version.split()[0], 'torch': torch.__version__, 'numpy': numpy.__version__, 'protobuf': google.protobuf.__version__})


In [ ]:
if not DATA_ROOT.exists():
    import kagglehub
    downloaded = pathlib.Path(kagglehub.dataset_download('sayedgamal99/smoke-fire-detection-yolo'))
    candidates = [downloaded] + [p for p in downloaded.rglob('*') if p.is_dir()]
    roots = [p for p in candidates if all((p/s/'images').is_dir() and (p/s/'labels').is_dir() for s in ('train', 'val', 'test'))]
    assert len(roots) == 1, f'Estrutura do espelho não reconhecida; organize train/val/test/images/labels. Conteúdo: {list(downloaded.iterdir())[:20]}'
    DATA_ROOT = roots[0]
print('Dados:', DATA_ROOT)


## 1. Limpar e auditar dados
O espelho Kaggle contém 397 caixas inválidas: 379 cruzam levemente a borda e 18 têm área zero. A limpeza reproduzível cria uma cópia derivada, recorta as 379 caixas aos limites e remove as 18 sem área. Cada alteração aparece em `cleaning.csv`; os originais permanecem intactos. A auditoria rigorosa valida a cópia. Abra alguns pares imagem/rótulo de cada classe e confirme visualmente se `0=fumaça, 1=fogo`.


In [ ]:
CLEAN = pathlib.Path('/content/dfire_clean')
if not CLEAN.exists():
    subprocess.run([sys.executable, str(PROJECT/'src/clean_dfire.py'), '--root', str(DATA_ROOT), '--output', str(CLEAN)], check=True)
DATA_ROOT = CLEAN
print('Dados preparados:', DATA_ROOT)


In [ ]:
CLASS_ZERO = 'fumaça'  # confirme visualmente
CLASS_ONE = 'fogo'    # confirme visualmente
AUDIT = pathlib.Path('/content/e2_artifacts/data_audit')
subprocess.run(['python', str(PROJECT/'src/prepare_dfire.py'), '--root', str(DATA_ROOT), '--output', str(AUDIT), '--class-zero', CLASS_ZERO, '--class-one', CLASS_ONE], check=True)
print((AUDIT/'summary.json').read_text())


## Subconjunto inicial de 3.000 imagens
A seleção é determinística (`seed=42`) e estratificada por imagens sem ocorrência, somente fumaça, somente fogo e ambas. Validação e teste permanecem completos.


In [ ]:
SUBSET = pathlib.Path('/content/dfire_e2_3000')
subprocess.run([sys.executable, str(PROJECT/'src/create_dfire_subset.py'), '--root', str(DATA_ROOT), '--output', str(SUBSET), '--size', '3000', '--seed', '42'], check=True)
SUBSET_YAML = SUBSET/'dfire_subset.yaml'
print((SUBSET/'summary.json').read_text())
print(SUBSET_YAML.read_text())


## 2. Baseline
Os pesos oficiais vêm da [release v0.1 do YOLOv7](https://github.com/WongKinYiu/yolov7/releases/tag/v0.1). A execução gera logs, opções, métricas, gráficos e checkpoint em `/content/e2_artifacts`. Copie o resultado para armazenamento persistente antes de encerrar a sessão. Não execute o teste isoladamente antes de fixar a configuração.


In [ ]:
WEIGHTS = pathlib.Path('/content/yolov7_training.pt')
if not WEIGHTS.exists():
    subprocess.run(['wget', '-q', '-O', str(WEIGHTS), 'https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7_training.pt'], check=True)
assert WEIGHTS.stat().st_size > 1_000_000, 'Download dos pesos parece incompleto'
BASELINE = pathlib.Path('/content/e2_artifacts/e2_3000_baseline')
subprocess.run([sys.executable, str(PROJECT/'src/run_baseline.py'), '--yolov7', str(YOLO), '--data', str(SUBSET_YAML), '--weights', str(WEIGHTS), '--output', str(BASELINE), '--epochs', '3', '--batch-size', '8', '--img-size', '640', '--device', '0', '--freeze-layers', '50', '--run-name', 'dfire-baseline-3000'], check=True)
print((BASELINE/'test.log').read_text()[-6000:])


## 3. Fine-tuning inicial
Repete exatamente a configuração do baseline, mas libera todas as camadas (`freeze=0`). Três épocas comprovam a execução e produzem evidências iniciais; não substituem o treinamento principal da E3.


In [ ]:
FINETUNE = pathlib.Path('/content/e2_artifacts/e2_3000_finetune')
subprocess.run([sys.executable, str(PROJECT/'src/run_baseline.py'), '--yolov7', str(YOLO), '--data', str(SUBSET_YAML), '--weights', str(WEIGHTS), '--output', str(FINETUNE), '--epochs', '3', '--batch-size', '8', '--img-size', '640', '--device', '0', '--freeze-layers', '0', '--run-name', 'dfire-finetune-inicial-3000'], check=True)
print((FINETUNE/'test.log').read_text()[-6000:])


## Após a execução
Registre em `docs/experimentos.md` o commit do projeto e do YOLOv7, contagens reais, GPU, versões, duração, métricas por classe, erros e localização dos logs. Inclua no Git apenas evidências leves que possam ser redistribuídas. Se uma dependência antiga do YOLOv7 falhar no Colab atual, registre a mensagem e a correção exata antes de repetir o experimento.
